# 02 — Templates de prompt A e B (Semana 7)
Configuração A: imagem + pergunta. Configuração B: imagem + pergunta + evidências estruturadas do PatchCore em texto.
A única diferença entre A e B é o bloco de evidências; o bloco da pergunta é idêntico.

In [1]:
import sys
sys.path.insert(0, '..')
import json, os
os.chdir('..')
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from src.prompts import build_prompt_a, build_prompt_b, build_messages, build_question_block, EVIDENCE_HEADER

MODEL_ID = 'HuggingFaceTB/SmolVLM-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
evidences = json.load(open('experiments/exp002_evidencias.json'))
subset = json.load(open('data/subset_avaliacao.json'))
QUESTIONS = {q['id']: q['text'] for q in subset['questions']}
print(DEVICE, len(evidences))

/home/edryck/Documentos/faculdade/ic/ic-visionexplain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/edryck/Documentos/faculdade/ic/ic-visionexplain/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/edryck/Documentos/faculdade/ic/ic-visionexplain/.venv/lib/python3.12/site-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

Loading weights:   4%|▎         | 23/657 [00:00<00:03, 186.70it/s]

Loading weights:   6%|▋         | 42/657 [00:00<00:04, 143.19it/s]

... (37 outputs left out)

Loading weights:  86%|████████▌ | 565/657 [00:07<00:00, 154.82it/s]

Loading weights:  93%|█████████▎| 609/657 [00:08<00:00, 225.56it/s]

Loading weights: 100%|██████████| 657/657 [00:08<00:00, 80.98it/s] 

cpu 5


In [2]:
sample = evidences[0]
question = QUESTIONS['q1_presence']
prompt_a = build_prompt_a(question)
prompt_b = build_prompt_b(question, sample)
print('--- A ---'); print(prompt_a)
print('--- B ---'); print(prompt_b)
assert prompt_b.endswith(prompt_a)
assert prompt_b.replace(prompt_a, '').strip().startswith(EVIDENCE_HEADER)

--- A ---
Question: Is there any defect in this image?
--- B ---
Automated inspection report (from an anomaly detection model):
The anomaly detector flagged this image as anomalous (score 0.947, threshold 0.5).
Region 1: lower center, covers 17.5% of the image, peak score 0.8155.

Question: Is there any defect in this image?


In [3]:
def answer(image_path, prompt_text, max_new_tokens=80):
    image = Image.open(image_path).convert('RGB')
    text = processor.apply_chat_template(build_messages(prompt_text), add_generation_prompt=True)
    inputs = processor(text=text, images=[image], return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()

question = QUESTIONS['q1_presence']
runs = []
for evidence in [evidences[0], evidences[2], evidences[4]]:
    a = answer(evidence['image'], build_prompt_a(question))
    b = answer(evidence['image'], build_prompt_b(question, evidence))
    runs.append({'image': evidence['image'], 'truth': evidence['ground_truth']['defect_type'], 'A': a, 'B': b})
    print(evidence['image'], '| truth:', evidence['ground_truth']['defect_type'])
    print('  A:', a)
    print('  B:', b)

data/mvtec_ad/mvtec-ad/bottle/test/broken_large/000.png | truth: broken_large
  A: Yes, there is a tear on the glass.
  B: Yes.


data/mvtec_ad/mvtec-ad/bottle/test/contamination/000.png | truth: contamination
  A: Yes, there is a reflection of an object in the middle of the image.
  B: Yes, there is a defect in this image.


data/mvtec_ad/mvtec-ad/bottle/test/good/001.png | truth: None
  A: Yes, there is a reflection of a light in the image.
  B: No.
